# Building a Growable Collection

CSC-239 · Module 5 · Lesson 3 of 3

You can control an object’s state through its methods. Now you will use a private array inside a class and make room for more entries when that array fills.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Implement ordered addition using a private backing array and logical size.
- Grow storage without losing entries and test the exact capacity boundary.


## Why This Matters

A signup list may begin with two names and later hold many more. Existing entries should keep their order when the list needs additional space.


## Check Your Starting Point

Explain why replacing an array element does not change array length. Recall how a for loop copies values one position at a time, how a reference variable can be reassigned, and why a caller should use public methods to reach private state.

**My explanation:**


## Concept

### Track used entries separately from available space

A collection’s **logical size** is the number of elements it currently uses. Its **capacity** is the number of positions available in its current backing array. These numbers serve different purposes.

An array always has its fixed length. A growable collection can use only some of those positions and remember the used count in a separate field. A new signup list might have capacity two and logical size zero.

| State | Logical size | Capacity | Used entries |
|---|---|---|---|
| New collection | 0 | 2 | None |
| After adding Ana | 1 | 2 | Ana |
| After adding Bo | 2 | 2 | Ana, Bo |
| After growing and adding Eli | 3 | 4 | Ana, Bo, Eli |

The remaining position after the last row is available space. It is not an added name. A traversal of the collection should stop at its logical size, even when the backing array is longer.

### Allocate an array with room for later values

**Array allocation** creates an array with a specified length. You have used brace initializers that supply every starting value. The expression new String[2] instead requests two positions for String references.

Those array elements receive **default values**, meaning the values Java supplies when it creates the array. String reference elements start as null; int elements start as zero. This differs from a local variable that you must assign before reading it.

```java
String[] labels = new String[2];
int[] counts = new int[2];
System.out.println(labels.length);
System.out.println(labels[0] == null);
System.out.println(counts[0]);
```

The output is 2, true, and 0. The null reference means no String object is stored there yet. Do not call a String method through that null value. An int element’s zero is a real numeric value, so zero alone cannot tell you whether a collection has added an element at that position. The separate logical size supplies that information.

### Keep backing storage inside the class

**Backing storage** is the internal array that holds a collection’s elements. Our GrowingNames class keeps a private String[] storage field and a private int size field. Its constructor allocates two positions and sets the logical size to zero.

The class exposes add, get, size, and capacity operations. The size reader returns the used count; the capacity reader returns the backing array’s length. These reader methods help you inspect the classroom model. They are not a claim about the exact graded assignment interface.

The state rule is 0 <= size and size <= storage.length. Positions below size contain the entries in their insertion order. Positions from size onward are unused capacity.

The get operation has a stated precondition: callers provide an index at least zero and less than the logical size. An unused backing-array position is not a valid collection element, even when that position exists in the array. You will practice checked rejection in the exception module; this lesson’s runnable calls satisfy the precondition.

### Grow by creating and copying

**Growth by copying** means allocating a larger array, copying the used elements in order, and assigning the storage field to the new array. The old array’s length does not change. The field now refers to a different array.

This complete short program demonstrates the copying mechanism:

```java
String[] storage = {"Ana", "Bo"};
int size = 2;
String[] larger = new String[storage.length * 2];
for (int index = 0; index < size; index = index + 1) {
    larger[index] = storage[index];
}
storage = larger;
System.out.println("Capacity: " + storage.length);
for (int index = 0; index < size; index = index + 1) {
    System.out.println(storage[index]);
}
```

The output is Capacity: 4, Ana, and Bo. The array with two positions remains a two-position array. The storage variable is reassigned to the new four-position array. Copying each used index to the same index preserves order.

The initial capacity in this lesson is positive: two. Doubling that number makes room for more entries. Growth is part of the class’s add operation so callers can request an addition without managing internal arrays.

### Grow before storing the new entry

The add operation follows this order:

1. If size equals capacity, allocate a larger array and copy the existing used entries.
2. Store the new entry at index size, which is the first unused position.
3. Increase size by one.

Checking first matters. If a full two-position array has size two, its next desired position is index two. The class must make that position available before storing the new entry.

Increasing size last matters too. The new entry belongs at the old size. Increasing first would skip the intended position and could select a position beyond capacity.

After two additions, the classroom collection has size two and capacity two. Its third addition grows capacity to four and then sets size to three. Its fifth addition grows capacity to eight. Growth creates spare positions but does not count those positions as added elements.

### Test the boundary and the order

Use a new collection for each test so earlier additions do not affect the starting state. Check these points:

- A new collection has size zero and capacity two; it has no valid get index.
- Exactly two additions fill the first array while preserving their order.
- The third addition triggers growth and keeps all three entries in order.
- Later growth keeps every earlier entry and appends the new one after them.

For GrowingScores in the independent task, zero is an allowed added score. The logical size must distinguish that real entry from unused zero-filled capacity. Traverse indexes below size and use get to compute the total. Explain the size and capacity separately when you interpret the result.

GrowingNames and GrowingScores are tutorial classes. The course assignment asks for StudentArrayList implementing the supplied SimpleArrayList interface across Modules 5 and 6. Follow the instructor’s exact interface and requirements when you work on that assignment; do not rename this tutorial class and assume its API is the required one.


## Video Demonstration

Follow the constructor, the full-array check and the copy loop. Watch the third name arrive after the original two names have moved into larger storage.

<video controls preload="metadata" width="960">
  <source src="media/03_building_a_growable_collection/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_building_a_growable_collection/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the building a growable collection demonstration transcript](media/03_building_a_growable_collection/transcript.md).


## Worked Example

**Subgoal 1: create the collection state.** Allocate two String positions and begin with zero used entries.

**Subgoal 2: preserve existing names when full.** Copy used positions to larger storage before adding another entry.

**Subgoal 3: report logical contents.** Print size and capacity at the boundaries, then read only the used entries.


In [ ]:
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
System.out.println("Start: " + names.size() + "/" + names.capacity());
names.add("Maya");
names.add("Luis");
System.out.println("Full: " + names.size() + "/" + names.capacity());
names.add("Nora");
System.out.println("Grown: " + names.size() + "/" + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}


Expected output:

```text
Start: 0/2
Full: 2/2
Grown: 3/4
Maya
Luis
Nora
```

The first two additions fit the original array. Before storing the third name at index 2, add creates a four-position array and copies indexes 0 and 1 into it. The new name follows those copied entries. Logical size becomes 3 while capacity is 4; traversal stops at size and prints only the three added names.


## Predict, Run, Trace, and Explain

### Predict a full collection after growth

Read the complete program without running it. Predict every output line, including the names in order. Track size and capacity after each addition. Mark the addition that needs a larger array and explain why the following addition either grows or fits. Record your prediction before running the next cell or opening the answer.

My predicted complete output:

Size and capacity after each addition:

The addition that triggers growth:

Why the final addition grows or fits:

The indexes the final loop may read:


In [ ]:
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}


Run the complete cell once and compare every line with your recorded prediction. Keep your original answer. If a line differs, trace the constructor, full-array condition, copy loop and size update to explain the difference. The cell constructs a new collection and initializes its instance fields each time it runs; run the full cell when checking a new attempt.

My original prediction:

My actual complete output:

Which lines agree or differ:

The first state change that explains any difference:

Why the earlier names keep their order:

What I changed in my explanation after running:

### Trace used positions and spare capacity

Trace the prediction program from the constructor through all four additions. For each row, record size, capacity, added entries and the valid get indexes. For Bea, describe the state immediately before the growth check, after copying but before insertion, and after insertion. Explain why growth alone does not increase size, why the old array does not change length, and why the new name is stored before size increases. Identify the private fields and the public operations used by the caller. Then predict and run the separate array-allocation check below.

| Point | Size | Capacity | Added entries in order | Valid get indexes |
|---|---|---|---|---|
| After constructor | | | | |
| After Iris | | | | |
| After Owen | | | | |
| After copying for Bea, before insertion | | | | |
| After Bea | | | | |
| After Kai | | | | |


Why copying does not itself add an element:

Why storage = larger does not stretch the old array:

Why the insertion uses the old size:

The private fields and the caller’s public operations:

Why get at index size is outside the collection’s valid-input rule even if spare array capacity exists:

How my trace explains the actual output:

<details>
<summary>Show answer</summary>

The constructor creates a two-position backing array and sets size to 0. Iris uses index 0 and Owen uses index 1. Before adding Bea, size equals capacity at 2, so add creates a four-position array and copies Iris and Owen to the same indexes. Bea goes at index 2 and size becomes 3. Kai fits at index 3 without another growth. Size and capacity are both 4. The loop reads indexes 0 through 3, which are exactly the four added elements. Every original array keeps its length; assigning storage = larger changes which array the field refers to. The next valid add will grow before writing at index 4. Immediately after copying, size is still 2 and capacity is 4; only Iris and Owen are collection entries. After Bea is stored, size becomes 3. An empty collection has no valid get index. Private storage remains inside the class; the public readers report its size, capacity or a valid logical element.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 4
Capacity: 4
Iris
Owen
Bea
Kai
```

Common error: Treating capacity as the number of names already added. Growing as soon as the array becomes full, instead of before the next addition when full. Assuming the new array already contains the earlier names.

</details>


### Distinguish a default value from an added element

Predict all five lines before running this complete array program. Explain which positions the program explicitly assigns and which keep Java’s default values. Then run it and compare. Why can the two printed integer zeros not tell you which position was explicitly assigned? Connect that limitation to the separate size field in a growable collection. This probe uses raw arrays; its valid array indexes do not make unused positions valid collection elements. Do not call a String method through a null reference.

My predicted complete output:

My actual complete output:

Which reference position still contains null:

Which positions were explicitly assigned:

Why the equal integer values do not reveal assignment history:

How a separate logical size identifies added entries:


In [ ]:
String[] labels = new String[3];
int[] counts = new int[3];
labels[1] = "ready";
counts[1] = 0;
System.out.println("Slots: " + labels.length);
System.out.println("First missing: " + (labels[0] == null));
System.out.println("Second: " + labels[1]);
System.out.println("First count: " + counts[0]);
System.out.println("Second count: " + counts[1]);


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

Each new array has length 3. The String array starts with null in every position; assigning ready at index 1 leaves index 0 null. The int array starts with zero in every position. Explicitly assigning zero at index 1 leaves it numerically equal to the untouched default at index 0. The values alone cannot tell you which assignment occurred. A growable collection therefore tracks its added entries with a separate size instead of searching for null or zero. The boolean comparison checks for null without calling a method through it.

```java
String[] labels = new String[3];
int[] counts = new int[3];
labels[1] = "ready";
counts[1] = 0;
System.out.println("Slots: " + labels.length);
System.out.println("First missing: " + (labels[0] == null));
System.out.println("Second: " + labels[1]);
System.out.println("First count: " + counts[0]);
System.out.println("Second count: " + counts[1]);
```

Expected output:

```text
Slots: 3
First missing: true
Second: ready
First count: 0
Second count: 0
```

Common error: Assuming new String[3] fills positions with empty strings. Assuming a zero value proves no score has been added. Using a default value as a replacement for the logical size.

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete a copy-and-append sequence

The displayed draft is incomplete and for reading only. Copy it into the empty work cell and replace all four placeholders. Choose COPY_TARGET from larger or storage. Choose COPY_SOURCE from storage or larger. Choose NEXT_STORAGE from larger or storage. Choose INSERT_INDEX from size or size + 1. Keep the rest unchanged. The completed program must preserve red and blue in that order and add green after them. Predict the size, capacity and three displayed values, run the complete repair, and explain the copy direction and the order of reference reassignment, insertion and size update.

This sample is for repair:

```java
String[] storage = {"red", "blue"};
int size = 2;
String[] larger = new String[storage.length * 2];
for (int index = 0; index < size; index = index + 1) {
    <COPY_TARGET>[index] = <COPY_SOURCE>[index];
}
storage = <NEXT_STORAGE>;
storage[<INSERT_INDEX>] = "green";
size = size + 1;
System.out.println("Size: " + size);
System.out.println("Capacity: " + storage.length);
for (int index = 0; index < size; index = index + 1) {
    System.out.println(storage[index]);
}
```


My four choices:

My predicted complete output:

My actual complete output:

Where each earlier value is copied:

Why the reference changes before insertion:

Why insertion uses size before it is increased:

Which allocated position remains unused:

<details>
<summary>Show answer</summary>

COPY_TARGET is larger and COPY_SOURCE is storage: the existing values must move from the old array into the new array. NEXT_STORAGE is larger so later insertion uses the array with four positions. INSERT_INDEX is size, which is 2 before insertion. Green goes after red and blue at index 2. Increasing size afterward gives 3 while capacity stays 4. The final loop visits only the three used positions and leaves the spare default-null position unprinted.

```java
String[] storage = {"red", "blue"};
int size = 2;
String[] larger = new String[storage.length * 2];
for (int index = 0; index < size; index = index + 1) {
    larger[index] = storage[index];
}
storage = larger;
storage[size] = "green";
size = size + 1;
System.out.println("Size: " + size);
System.out.println("Capacity: " + storage.length);
for (int index = 0; index < size; index = index + 1) {
    System.out.println(storage[index]);
}
```

Expected output:

```text
Size: 3
Capacity: 4
red
blue
green
```

Common error: Copying default nulls from the new array over earlier values. Keeping storage attached to the full old array. Using size + 1 for insertion and skipping the next position.

</details>


### Add one entry beyond the new capacity

Run this complete four-name starter and record its report. Insert only `names.add("Zoe");` after the Kai addition and before the two report statements. Keep the class and traversal unchanged. Predict the new complete output before running the whole modified cell. Trace which entries are copied and where Zoe is stored. Explain why the call causes a growth even though the caller does not create an array itself. Remove only the Zoe addition and run the complete restored starter to compare the fourth-addition boundary again.


In [ ]:
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}


My starter output:

My predicted modified output:

My actual modified output:

Size and capacity just before the fifth addition:

The indexes copied during growth:

The index used for Zoe:

Why spare positions are not printed:

My output after restoring four additions:

How I ensured each run began with a new collection:

<details>
<summary>Show answer</summary>

Before the fifth addition, size and capacity are both 4. Adding Zoe triggers allocation of an eight-position array. The loop copies Iris, Owen, Bea and Kai to indexes 0 through 3 in order. Storage then refers to the new array, Zoe is stored at index 4, and size becomes 5. The three remaining positions are spare capacity. No caller-side array work is needed because add contains the growth rule. Restoring the four-addition caller constructs a fresh collection whose final size and capacity are both 4.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
names.add("Zoe");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 5
Capacity: 8
Iris
Owen
Bea
Kai
Zoe
```

Common error: Changing the constructor’s initial capacity instead of exercising growth. Adding Zoe after the report and expecting the earlier report to include her. Reporting all eight capacity positions as collection entries.

**Additional test: `Four additions: Iris, Owen, Bea, Kai`.** The fourth entry fits the four-position array produced by the third addition.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 4
Capacity: 4
Iris
Owen
Bea
Kai
```

</details>


### Repair copying in the wrong direction

The displayed program is intentionally incorrect; do not run it. Its copy assignment is reversed. Trace the third addition and predict the complete faulty report, including the names. Identify which values are overwritten and where the null values come from. Write the complete repaired program in the empty work cell. Change only the copy assignment so each old entry is copied into the same index in larger. Keep the rest of the class, four names and report unchanged. Predict and run the repair. Also test the complete repaired program with only Iris, Owen and Bea, then with no add calls. Keep the report and traversal. Explain why size and capacity alone would miss this copying bug. Restore all four names afterward.

This sample is for repair:

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                storage[index] = larger[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```


My predicted faulty complete output:

The old and new arrays just before the first copy:

What the reversed assignment does at indexes 0 and 1:

My repaired assignment:

My predicted and actual repaired four-name output:

My predicted and actual three-name output:

My predicted and actual empty output:

Why size and capacity alone miss this defect:

Why the empty case must not call get:

My actual output after restoring the four names:

<details>
<summary>Show answer</summary>

During the third addition, the faulty loop reads null from the newly allocated larger array and writes it over Iris and Owen in storage. It then assigns storage = larger, whose first two positions still hold null. Bea and Kai are added at indexes 2 and 3, so the faulty report has the expected size 4 and capacity 4 but prints null, null, Bea and Kai. Repair the assignment to larger[index] = storage[index]. The loop now preserves Iris and Owen, and all four names appear in order. The three-name case checks the first growth directly. The empty case has size 0, capacity 2 and no valid get index; the traversal performs zero iterations. Encapsulation does not make an incorrect internal copy correct.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 4
Capacity: 4
Iris
Owen
Bea
Kai
```

Common error: Reversing the reference assignment instead of the element-copy assignment. Treating correct size and capacity as proof that values survived. Testing only two additions, which never execute the copy loop. Calling get(0) on an empty collection to inspect its unused array position.

**Additional test: `Repaired caller with Iris, Owen and Bea only`.** The third addition triggers the first growth and preserves both earlier names.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 3
Capacity: 4
Iris
Owen
Bea
```

**Additional test: `Repaired caller with no additions`.** No entry was added. The logical traversal has zero iterations and makes no get call.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 0
Capacity: 2
```

</details>


## Independent Practice

### Build GrowingScores with private backing storage

Build the tutorial class GrowingScores. Use a private int[] storage field and a separate private int size field. Its public no-argument constructor must allocate capacity 2 and initialize size to 0. Write public void add(int score), public int get(int index), public int size(), and public int capacity(). When full, add must allocate an array with twice the current length, copy the used values in order, and replace storage before inserting the new score at index size. Increase size after insertion. The get caller must provide 0 <= index < size; use only valid indexes and do not add exception handling. Create scores, add 4, 0, 7, 2 and 5 in that order, then use a loop over logical indexes to compute total. Print exactly `Size: 5`, `Capacity: 8`, and `Total: 18`. Plan the fields and state rule before writing. Predict the result, run your entire class and caller so each attempt constructs a new collection, and explain why the added zero counts while unused zeros do not. This tutorial class is separate from the graded StudentArrayList assignment; do not invent the missing SimpleArrayList interface.

My fields and their different purposes:

My constructor initialization:

My rule relating size and capacity:

My growth, copy and insertion order:

My get precondition and valid loop bound:

My predicted complete output:

My actual complete output:

Why index 1 contains an added score even though its value is zero:

Why unused zero positions are outside the logical collection:

Why the fifth addition keeps the first four values:


### Test empty, full and grown collections

Test every row in the table with your complete GrowingScores class and caller. Keep the class, logical traversal and three report statements unchanged; change only the add calls. Run the whole cell for each case so its constructor creates a new collection, rather than adding onto a previous test. Predict all three report lines before each run. Afterward record the actual output and explain each size/capacity boundary. Compare the empty case with the two-zero case: both totals can be zero while logical sizes differ. Then restore the exact five additions. After its three report lines, add a second loop over indexes below scores.size() that prints `Score `, the index, `: ` and scores.get(index). Predict and compare all five indexed values in insertion order. Count exactly five diagnostic lines, ending at index 4, and check that no indexes 5 through 7 are reported. Adding unused zero-filled positions could leave Total: 18 unchanged, so that total alone does not prove the traversal bound is correct. Explain why a correct sum alone cannot prove the order or show whether a zero was counted. Repair any mismatch and repeat the cases. Remove the diagnostic loop when finished and restore the exact contracted five-addition output. Never call get for a negative index, an index equal to size, or any index on an empty collection.

| Added scores, in order | Predicted Size / Capacity / Total | Actual three report lines | Match or repair |
|---|---|---|---|
| None | | | |
| 4, 0 | | | |
| 4, 0, 7 | | | |
| 4, 0, 7, 2 | | | |
| 4, 0, 7, 2, 5 | | | |
| 0, 0 | | | |


My predicted five indexed diagnostic lines:

My actual five indexed diagnostic lines:

My counted number of diagnostic lines and final index:

Why including unused zeros could leave Total: 18 unchanged:

Why growth happens before the third and fifth additions:

Why the second and fourth additions fit without growth:

Why empty and two-zero totals can agree while sizes differ:

Why neither a total nor spare capacity proves the order of added values:

How each test creates a new collection:

My correction and the cases I repeated:

My actual output after removing the diagnostic and restoring the contracted program:


<details>
<summary>Show answer</summary>

GrowingScores keeps the backing int array and logical size private. Its constructor allocates two positions and explicitly sets size to 0. The five additions put 4, 0, 7, 2 and 5 at logical indexes 0 through 4. The third addition copies the first two scores into capacity 4; the fifth copies the first four into capacity 8. Each new score is stored at the old size, then size increases by one. The caller reads only indexes below scores.size(), so total is 4 + 0 + 7 + 2 + 5 = 18. The added zero at index 1 counts as an element. The unused zero-filled positions at indexes 5 through 7 do not count, because they lie at or beyond logical size. A total alone cannot reveal whether the zero was counted or whether entries were reordered; check size and the ordered values too. The get method assumes 0 <= index < size and does not enforce that rule itself. Summing the three unused zero-filled array positions could also leave 18 unchanged. The valid diagnostic prints exactly five entries at indexes 0 through 4, so its line count and final index provide evidence about the logical traversal bound. The boundary table checks a new collection, a full initial array, the first growth, a full four-position array and the second growth. The two-zero case separates the count of additions from the total. The indexed report checks the values and insertion order after both copies; size, capacity and total alone are not enough to establish all three.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
scores.add(7);
scores.add(2);
scores.add(5);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 5
Capacity: 8
Total: 18
```

Common error: Using capacity as the traversal limit. Skipping an added score because its value is zero. Returning storage.length from size(). Appending before making space in a full array. Assuming the tutorial operations define the graded assignment interface.

**Additional test: No additions.** The constructor reserves two positions but adds no elements. The loop has zero iterations and makes no get call.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 0
Capacity: 2
Total: 0
```

**Additional test: Add 4, then 0.** Both additions fit. The explicit zero is still an added element, so size is 2.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 2
Capacity: 2
Total: 4
```

**Additional test: Add 4, 0, 7.** The third addition grows before writing at index 2 and preserves 4 and 0.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
scores.add(7);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 3
Capacity: 4
Total: 11
```

**Additional test: Add 4, 0, 7, 2.** The fourth addition fills the four-position array; it does not yet require another allocation.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
scores.add(7);
scores.add(2);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 4
Capacity: 4
Total: 13
```

**Additional test: Add 0, then 0.** Two zero additions have the same total as no additions, but their logical size is 2 rather than 0.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(0);
scores.add(0);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 2
Capacity: 2
Total: 0
```

**Additional test: Exact contracted program followed by the indexed diagnostic loop.** The five valid get calls establish insertion order and the explicitly added zero after both growth events. The extra diagnostic is removed to restore the contracted three-line report. Exactly five indexed lines end at index 4; indexes 5 through 7 must not appear. Adding their unused zeros would not change the total, so the sum is insufficient evidence.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
scores.add(7);
scores.add(2);
scores.add(5);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
for (int index = 0; index < scores.size(); index = index + 1) {
    System.out.println("Score " + index + ": " + scores.get(index));
}
```

Expected output:

```text
Size: 5
Capacity: 8
Total: 18
Score 0: 4
Score 1: 0
Score 2: 7
Score 3: 2
Score 4: 5
```

</details>


## Summary

Logical size counts used elements; capacity counts available array positions. New arrays receive default element values, so unused capacity needs a separate used count. A growing collection allocates larger storage, copies used entries in order, and then appends the new entry. Each individual array keeps its fixed length.

Close the answers and trace size, capacity and valid collection indexes before and after the first growth.


## Reflection

Explain how a growing signup list can preserve the order students joined. Describe tests for an empty list, a full backing array and one new entry beyond that boundary. State what a caller needs to know about valid indexes.

**My design and explanation:**

Module 6 will give reusable classes explicit interface contracts and explore shared behavior through inheritance.


## Supplemental Reading

- [Java 21 arrays](https://docs.oracle.com/javase/specs/jls/se21/html/jls-10.html) defines array creation, fixed length and element access.
- [Java 21 initial values](https://docs.oracle.com/javase/specs/jls/se21/html/jls-4.html#jls-4.12.5) specifies default array-element values and distinguishes them from local-variable initialization.
